# Week 1 — SELECT, WHERE, ORDER BY, LIMIT: Reading Data
## Phase 2b SQL | PORA Academy Cohort 7 — **Demo**

By the end of this session, you will be able to:
- Write SELECT with WHERE, ORDER BY, and LIMIT from memory

Today we start reading data with raw SQL against the same Olist dataset you already
know from Phase 2a — except this time you'll talk to the database directly instead
of going through pandas.

### Setup — run this cell first

This loads the 8 Olist tables into a file-based SQLite database and connects the
`%%sql` cell magic to it. Because `autopandas` is on, every `%%sql` query returns a
pandas DataFrame under the hood — useful later when we want to inspect a result in
Python — but for now we'll mostly just look at the tables SQL prints back to us.

In [ ]:
# =====================================================================
# Olist SQL Setup — runs on BOTH Google Colab and a local machine.
# Run this cell FIRST. It loads the 8 Olist tables into a SQLite
# database and connects the %%sql magic to it. You should not need to
# edit anything unless auto-detection fails (see the two knobs below).
#
# Design notes:
# - We teach SQL with the %%sql cell magic (jupysql), not pd.read_sql().
# - jupysql opens its OWN connection, so the DB must be a real FILE
#   (a :memory: DB would be invisible to it).
# - We use jupysql (the maintained SQL magic). On Colab we install it,
#   because Colab ships the legacy ipython-sql, which (a) can't take a
#   connection by engine variable and (b) renders every result through
#   prettytable.__dict__[style], crashing on modern prettytable with
#   KeyError 'DEFAULT'/'SINGLE_BORDER'. jupysql fixes both.
# - autopandas=True makes every %%sql result a pandas DataFrame, which
#   lets the self-check cells assert on .iloc/.shape directly.
# =====================================================================
import os, glob, sqlite3, tempfile, zipfile
import pandas as pd

# --- Optional knobs (leave blank; only set if auto-detect fails) ------
LOCAL_DATA_DIR = ""   # local run: folder that holds olist_orders_dataset.csv
DRIVE_ZIP_PATH = ""   # Colab: full path to phase-2-python-sql.zip in your Drive
# ---------------------------------------------------------------------

# Detect Colab (google.colab only imports there). Outside Colab — including
# the content-pipeline validator — this falls through to the local branch.
try:
    from google.colab import drive
    drive.mount("/content/drive")
    ON_COLAB = True
except ModuleNotFoundError:
    ON_COLAB = False


def _colab_find_zip():
    """Locate phase-2-python-sql.zip in Drive WITHOUT a full recursive scan
    (globbing '/content/drive/MyDrive/**' walks the entire Drive over the
    network and can hang for many minutes). Try explicit paths first, then a
    depth- and count-bounded breadth-first search that prints progress."""
    if DRIVE_ZIP_PATH:
        if os.path.exists(DRIVE_ZIP_PATH):
            return DRIVE_ZIP_PATH
        raise FileNotFoundError(f"DRIVE_ZIP_PATH is set but not found: {DRIVE_ZIP_PATH}")

    target = "phase-2-python-sql.zip"
    # Fast, instant checks of the most likely spots (top of Drive + course folder).
    for cand in (
        f"/content/drive/MyDrive/{target}",
        f"/content/drive/MyDrive/Data Analysis and AI Automation Course Cohort 7/Dataset/{target}",
        f"/content/{target}",
    ):
        if os.path.exists(cand):
            return cand

    # Bounded BFS: depth <= 4, at most ~600 folders, skipping hidden dirs.
    print("Searching your Google Drive for phase-2-python-sql.zip ...")
    root, queue, scanned = "/content/drive/MyDrive", [("/content/drive/MyDrive", 0)], 0
    while queue:
        d, depth = queue.pop(0)
        hit = os.path.join(d, target)
        if os.path.exists(hit):
            return hit
        if depth >= 4:
            continue
        try:
            for e in os.scandir(d):
                if e.is_dir() and not e.name.startswith("."):
                    queue.append((e.path, depth + 1))
        except OSError:
            continue
        scanned += 1
        if scanned % 50 == 0:
            print(f"  ...scanned {scanned} folders")
        if scanned >= 600:
            break

    raise FileNotFoundError(
        "Could not quickly find phase-2-python-sql.zip in your Drive. Put the zip at the "
        "TOP of your Drive (My Drive) and re-run, or set DRIVE_ZIP_PATH at the top of this "
        "cell to its exact path.")


def _find_csv_dir():
    """Return the folder that actually contains olist_orders_dataset.csv."""
    roots = []
    env_dir = os.environ.get("OLIST_DATA_PATH", "")   # set by the pipeline validator
    if env_dir:
        roots.append(env_dir)
    if LOCAL_DATA_DIR:
        roots.append(LOCAL_DATA_DIR)

    if ON_COLAB:
        extract_path = "/content/olist_data"
        # unzip only the first time; reuse the extracted CSVs afterwards
        if not glob.glob(f"{extract_path}/**/olist_orders_dataset.csv", recursive=True):
            zip_path = _colab_find_zip()
            os.makedirs(extract_path, exist_ok=True)
            print(f"Unzipping {os.path.basename(zip_path)} ...")
            with zipfile.ZipFile(zip_path) as z:
                z.extractall(extract_path)
        roots.append(extract_path)
    else:
        # Local: search cwd (recursively) + a few common spots — never the whole
        # home dir (that recursive walk can be very slow). Set LOCAL_DATA_DIR if
        # your CSVs live elsewhere.
        roots += [os.getcwd(),
                  os.path.expanduser("~/Downloads"),
                  os.path.expanduser("~/Desktop"),
                  os.path.expanduser("~/olist")]

    for root in roots:
        if os.path.exists(os.path.join(root, "olist_orders_dataset.csv")):
            return root
        hits = glob.glob(os.path.join(root, "**", "olist_orders_dataset.csv"), recursive=True)
        if hits:
            return os.path.dirname(hits[0])

    raise FileNotFoundError(
        "Olist CSVs not found. Set LOCAL_DATA_DIR (local) or DRIVE_ZIP_PATH (Colab) at "
        "the top of this cell.")


DATA_DIR = _find_csv_dir()
print("Data folder:", DATA_DIR)

# Build a file-based SQLite DB shared by pandas (loading) and jupysql (querying).
DB_PATH = os.environ.get("OLIST_DB_PATH") or (
    "/content/olist.db" if ON_COLAB else os.path.join(tempfile.gettempdir(), "olist.db"))

tables = {
    "orders": "olist_orders_dataset.csv",
    "customers": "olist_customers_dataset.csv",
    "order_items": "olist_order_items_dataset.csv",
    "order_payments": "olist_order_payments_dataset.csv",
    "order_reviews": "olist_order_reviews_dataset.csv",
    "products": "olist_products_dataset.csv",
    "sellers": "olist_sellers_dataset.csv",
    "product_category_translation": "product_category_name_translation.csv",
}

conn = sqlite3.connect(DB_PATH)
for table_name, filename in tables.items():
    df = pd.read_csv(os.path.join(DATA_DIR, filename))
    df.to_sql(table_name, conn, if_exists="replace", index=False)
    print(f"Loaded {table_name}: {len(df):,} rows")
conn.close()
print("\nDatabase ready.")

# On Colab, install jupysql so `%load_ext sql` loads it instead of the legacy
# ipython-sql (see header). Off Colab (local / pipeline validator) jupysql is
# already installed, so we skip the install and stay offline-safe.
if ON_COLAB:
    get_ipython().run_line_magic("pip", "install --quiet --upgrade jupysql")

get_ipython().run_line_magic("load_ext", "sql")

# Guard: if the legacy ipython-sql was already loaded earlier THIS session (e.g.
# an older cell ran first), the freshly installed jupysql cannot hot-swap in — a
# runtime restart is the only fix. jupysql exposes sql.connection.ConnectionManager;
# ipython-sql does not. Stop with a clear instruction instead of a later cryptic
# prettytable KeyError.
import sql.connection as _sqlconn
if not hasattr(_sqlconn, "ConnectionManager"):
    raise RuntimeError(
        "Legacy ipython-sql is active, not jupysql. On Colab: Runtime -> Restart session, "
        "then run THIS setup cell first (before any other cell). Locally: "
        "pip install --upgrade jupysql and restart the kernel."
    )

# Connect the %%sql magic to the SAME database file. autopandas=True is REQUIRED
# (see header). We connect with run_line_magic (not a literal `%sql` line) so the
# computed DB_PATH is interpolated correctly. Do NOT set SqlMagic.style.
get_ipython().run_line_magic("config", "SqlMagic.autopandas = True")
get_ipython().run_line_magic("config", "SqlMagic.feedback = 0")
get_ipython().run_line_magic("sql", f"sqlite:///{DB_PATH}")

# Verify (expected row counts — do not alter without re-running against data):
#   orders 99,441 | customers 99,441 | order_items 112,650 | order_payments 103,886
#   order_reviews 99,224 | products 32,951 | sellers 3,095 | product_category_translation 71


## Why this matters

The `orders` table holds all **99,441** orders Olist has ever processed, but only
**96,478** of them were actually delivered to a customer — the rest were canceled,
lost, or are still in transit. If someone in ops asks *"how many orders never made it
to the customer?"*, scrolling through 99,441 rows in a spreadsheet is not an option.
`SELECT` and `WHERE` are how we ask the database that question directly and get back
exactly the rows — or the count — we need, in milliseconds. Today's four clauses
(`SELECT`, `WHERE`, `AND`/`OR`, `ORDER BY` + `LIMIT`) are the four moves you will use
in almost every query you ever write.

## 1. `SELECT` — choosing columns

`SELECT` is how you tell the database *which columns* you want back. `SELECT *`
means "give me every column, I haven't decided what I need yet" — it's the SQL
equivalent of opening a spreadsheet and scrolling right to see everything. Once you
know exactly which columns matter for the question you're answering, you name them
explicitly instead — this is faster for the database to return and much easier for a
human to read, especially on a table with a dozen-plus columns like `orders`. Think
of `SELECT *` as "show me the whole filing cabinet" and `SELECT col1, col2` as "just
hand me these two folders.

In [ ]:
%%sql
-- SELECT * : every column, useful for a first look at a table you don't know yet
SELECT *
FROM orders
LIMIT 10

In [ ]:
%%sql
-- Naming columns explicitly: only what we actually need for the question at hand
SELECT order_id, customer_id, order_status, order_purchase_timestamp
FROM orders
LIMIT 10

## 2. `WHERE` — filtering rows

`SELECT` picks columns; `WHERE` picks *rows*. It's a boolean test applied to every
row in the table — if the condition is true for that row, the row stays in the
result; if it's false, it's dropped. This is exactly like Excel's filter dropdown,
except instead of clicking checkboxes you write the condition as text:
`order_status = 'delivered'` reads almost like English. Below we filter to just the
delivered orders, then count how many rows survive the filter under a few different
conditions — these three counts are all independently verified against the dataset,
so if your query returns something else, the query (not the number) is wrong.

In [ ]:
%%sql
-- Filter to only delivered orders and look at a sample of the rows
SELECT order_id, customer_id, order_status
FROM orders
WHERE order_status = 'delivered'
LIMIT 10

In [ ]:
%%sql
-- How many orders are delivered, in total?
SELECT COUNT(*) AS total
FROM orders
WHERE order_status = 'delivered'   -- Expected: 96,478

In [ ]:
%%sql
-- != means "not equal to" — every order that is anything OTHER than delivered
SELECT COUNT(*) AS total
FROM orders
WHERE order_status != 'delivered'   -- Expected: 2,963

## 3. `AND` / `OR` in `WHERE`

Real business questions rarely stop at one condition. `AND` requires *every* listed
condition to be true for a row to survive — it narrows the result. `OR` requires
*at least one* of the listed conditions to be true — it widens the result to include
either group. A useful mental model: `AND` is the intersection of two filters, `OR`
is the union. Here we want every order that went wrong in one of two ways — canceled
or unavailable — so we reach for `OR`.

In [ ]:
%%sql
-- OR widens the filter: canceled orders UNION unavailable orders
SELECT COUNT(*) AS total
FROM orders
WHERE order_status = 'canceled'
   OR order_status = 'unavailable'   -- Expected: 1,234 (625 canceled + 609 unavailable)

## 4. `ORDER BY` and `LIMIT`

By default, a database returns rows in whatever order it happens to store them —
which is not useful when you want "the top 10 highest payments" or "the 5 oldest
orders." `ORDER BY column DESC` sorts rows from highest to lowest on that column
(`ASC`, ascending, is the default if you omit it); `LIMIT n` then caps the result to
just the first `n` rows after sorting. The two clauses almost always travel
together: sort first, then cut off at the number of rows you actually want to see.

In [ ]:
%%sql
-- Sort payments highest-first, keep only the top 10
SELECT order_id, payment_type, payment_value
FROM order_payments
ORDER BY payment_value DESC
LIMIT 10

## Going deeper — integer division silently truncates

SQLite (like most SQL) does *integer* division when both operands are whole numbers:
`5 / 2` gives `2`, not `2.5` — the decimal part is just thrown away. This bites
people the moment they try to compute a percentage. The fix is to force at least one
side to be a real (floating-point) number by multiplying by `1.0` before dividing.
Below, we compute delivered orders as a share of all orders using our two verified
base counts (96,478 delivered out of 99,441 total) — watch what changes when we add
`* 1.0`.

In [ ]:
%%sql
-- WRONG (commented out) — integer division truncates to 0:
--   SELECT (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered')
--          / (SELECT COUNT(*) FROM orders) AS delivered_share
-- CORRECT — multiply by 1.0 to force real division:
SELECT
  (SELECT COUNT(*) FROM orders WHERE order_status = 'delivered') * 1.0
  / (SELECT COUNT(*) FROM orders) AS delivered_share

## Common mistake — `= NULL` never matches

A beginner instinct is to filter for missing values with `column = NULL`. In SQL,
`NULL` means "unknown," and an unknown is never considered equal to anything — not
even to another `NULL`. `column = NULL` silently returns **zero rows every time**,
with no error to warn you. The correct way to test for missing data is the dedicated
`IS NULL` (or `IS NOT NULL`) operator. We can see this directly on
`order_delivered_customer_date`, which is `NULL` whenever an order was never actually
delivered to the customer.

In [ ]:
%%sql
-- ── COMMON MISTAKE ──────────────────────────────────
-- WRONG — matches ZERO rows, NULL is never '= NULL':
--   SELECT COUNT(*) FROM orders WHERE order_delivered_customer_date = NULL
-- CORRECT — use IS NULL:
SELECT COUNT(*) AS total
FROM orders
WHERE order_delivered_customer_date IS NULL   -- Expected: 2,965

## Mini-challenge — your turn

⏱ ~5 minutes

Using only `SELECT` and `WHERE`, count how many payments in `order_payments` were
made using `boleto` (a common Brazilian payment method, similar to a bank slip).

**Expected:** 19,784 boleto payments.

In [ ]:
%%sql
-- ⏱ ~5 min — your turn! Replace the placeholder below with your own query.
SELECT 'write your query here' AS todo

## Session Summary

| Clause | What it does | Example |
|---|---|---|
| `SELECT` | choose columns | `SELECT order_id, order_status FROM orders` |
| `WHERE` | filter rows | `WHERE order_status = 'delivered'` |
| `AND` | narrow — every condition must be true | `WHERE a = 1 AND b = 2` |
| `OR` | widen — at least one condition must be true | `WHERE order_status = 'canceled' OR order_status = 'unavailable'` |
| `ORDER BY` | sort rows (`ASC`/`DESC`) | `ORDER BY payment_value DESC` |
| `LIMIT` | cap the number of rows returned | `LIMIT 10` |
| `IS NULL` | test for missing values (never `= NULL`) | `WHERE order_delivered_customer_date IS NULL` |

---
**Coming up Thursday**: independent practice — you'll apply `SELECT`, `WHERE`,
`ORDER BY`, and `LIMIT` on your own, without a demo to follow, to answer business
questions about Olist orders, payments, and reviews (group work).

Thursday is a **group exercise**: working in small groups, you'll apply today's
clauses to answer 5 business questions — NULL delivery dates, the top-5 priced
items, boleto payments, SP-state customers, and the worst (1-star) reviews —
using only what you learned today.